## Configurations

In [1]:
from pathlib import Path
from typing import Final

# ── LLM & Embedding ──────────────────────────────────────────────
MODEL_NAME: Final[str] = "llama3.2:3b"
EMBEDDING_MODEL: Final[str] = "nomic-embed-text:latest"
EMBEDDING_DIM = 768
# ── Directories (relative to project root) ───────────────────────
ROOT_DIR: Final[Path] = Path('.').resolve()
print(ROOT_DIR)
DATA_DIR: Final[Path] = ROOT_DIR / "data"
LOG_DIR: Final[Path] = ROOT_DIR / "logs"
GRAPH_DIR: Final[Path] = ROOT_DIR / "graphs"
HISTORY_DIR: Final[Path] = ROOT_DIR / "history"

# ── Chunking & Retrieval settings ────────────────────────────────
CHUNK_SIZE: Final[int] = 800
CHUNK_OVERLAP: Final[int] = 150
TOP_K: Final[int] = 7
MAX_SOURCE_LENGTH: Final[int] = 250  # used in fact extraction

# Ensure all directories exist
for directory in (DATA_DIR, LOG_DIR, GRAPH_DIR, HISTORY_DIR):
    directory.mkdir(parents=True, exist_ok=True)


DB_CONFIG = {
    'host': 'localhost',
    'port': 5433,
    'dbname': 'personalagent',
    'user': 'postgres',
    'password': 'Ganesh123'
}

/home/pawankrgunjan/Development/GenAI/GenerativeAI/PersonalAgent


## Logger

In [2]:
import logging
from datetime import datetime
from pathlib import Path

from colorama import Fore, Style, init

init(autoreset=True)


def setup_logger(user_id: str) -> logging.Logger:
    """
    Create and configure a per-user logger.

    - Console: colored, INFO level
    - File: detailed with tracebacks, DEBUG level
    """
    logger = logging.getLogger(f"agent.{user_id}")
    logger.setLevel(logging.DEBUG)  # capture everything in file

    # Prevent duplicate handlers if called multiple times
    if logger.hasHandlers():
        logger.handlers.clear()

    # ── Console handler ────────────────────────────────────────────────
    console_handler = logging.StreamHandler()
    console_handler.setLevel(logging.INFO)

    # Use formatter with colorama codes (no monkey-patch needed)
    console_formatter = logging.Formatter(
        f"{Fore.CYAN}%(asctime)s{Style.RESET_ALL} "
        f"[{Fore.GREEN}%(levelname)s{Style.RESET_ALL}] "
        f"%(message)s",
        datefmt="%H:%M:%S",
    )
    console_handler.setFormatter(console_formatter)

    # ── File handler ───────────────────────────────────────────────────
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    log_file: Path = LOG_DIR / f"agent_{user_id}_{timestamp}.log"

    file_handler = logging.FileHandler(log_file, encoding="utf-8")
    file_handler.setLevel(logging.DEBUG)

    file_formatter = logging.Formatter(
        "%(asctime)s [%(levelname)s] %(name)s:%(lineno)d - %(message)s\n"
        "%(exc_info)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )
    file_handler.setFormatter(file_formatter)

    # ── Add handlers and finalize ──────────────────────────────────────
    logger.addHandler(console_handler)
    logger.addHandler(file_handler)
    logger.propagate = False

    logger.info("Logger initialized → %s", log_file)
    return logger


In [3]:
logger = setup_logger('PersonalAgent')

09:21:56 [INFO] Logger initialized → /home/pawankrgunjan/Development/GenAI/GenerativeAI/PersonalAgent/logs/agent_PersonalAgent_20260227_092156.log


## Schemas

In [4]:
from typing import List
from pydantic import BaseModel, Field

class UserFact(BaseModel):
    key: str = Field(..., description="Fact key (name, location, salary, doj, etc.)")
    value: str = Field(..., description="Exact literal value from text")
    confidence: float = Field(1.0, ge=0.0, le=1.0, description="1.0 for obvious facts, lower if uncertain")
    source_text: str = Field(..., description="Snippet of original text proving the fact")

class ExtractedFacts(BaseModel):
    facts: List[UserFact] = Field(default_factory=list, description="All extracted facts; empty if none")

## LLM

In [5]:
from typing import Any
import numpy as np
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import Runnable
from langchain_ollama import ChatOllama, OllamaEmbeddings

In [6]:
def get_llm(temperature: float = 0.0, **kwargs: Any) -> ChatOllama:
    return ChatOllama(model=MODEL_NAME, temperature=temperature, **kwargs)

def get_embeddings() -> OllamaEmbeddings:
    return OllamaEmbeddings(model=EMBEDDING_MODEL)

In [7]:
llm: ChatOllama = get_llm(temperature=0.0)
embeddings_model: OllamaEmbeddings = get_embeddings()

In [8]:
def embed_text(text: str) -> np.ndarray:
    """
    Embed text using Ollama embeddings (nomic-embed-text).
    Returns numpy float32 vector of fixed dimension (EMBEDDING_DIM).
    """
    try:
        vec = embeddings_model.embed_query(text)  # returns list[float] [web:35]
        if len(vec) != EMBEDDING_DIM:
            raise ValueError(f"Expected {EMBEDDING_DIM} dims, got {len(vec)}")
        return np.asarray(vec, dtype=np.float32)
    except Exception as exc:
        raise RuntimeError(f"Embedding failed: {exc}") from exc

In [9]:

#llm = ChatOllama(model=MODEL_NAME, temperature=0.0, format="json")

extraction_prompt = ChatPromptTemplate.from_messages([
    ("system", """Extract explicit facts as key-value pairs. Use confidence=1.0 for obvious facts.

Examples (exact format):
- "I am Rupam" → key="name", value="Rupam", confidence=1.0  
- "I live in Patna" → key="location", value="Patna", confidence=1.0

For "Hi, I am Rupam & You?": ONLY extract name="Rupam"
Ignore "You?", punctuation.

Output ONLY valid JSON array matching schema."""),
    ("human", "{message}")
])

extraction_chain = extraction_prompt | llm.with_structured_output(ExtractedFacts)

In [10]:
# Test
last_human = "Hi, I am Rupam & You?"
extracted = extraction_chain.invoke({"message": last_human})
print(extracted.facts)

[UserFact(key='name', value='Rupam', confidence=1.0, source_text='I am Rupam')]


## Document Reader

In [11]:
import pandas as pd
import hashlib
from pathlib import Path
from typing import Optional
from langchain_community.document_loaders import CSVLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
#from langchain_experimental.text_splitter import SemanticChunker

In [12]:
#text_splitter = SemanticChunker(embeddings_model, buffer_size=50, breakpoint_threshold_type='standard_deviation')

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    length_function=len,
    separators=["\n\n", "\n", " ", ""],
)

In [13]:
def compute_file_hash(filepath: Path | str,log: Optional[logging.Logger]) -> str:
    filepath = Path(filepath)
    hasher = hashlib.sha256()
    log.info(f'Opening {filepath}')
    with filepath.open("rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            hasher.update(chunk)
    return hasher.hexdigest()

In [14]:
resume_path = DATA_DIR/'pawankrgunjan.pdf'

compute_file_hash(resume_path, logger)

09:22:27 [INFO] Opening /home/pawankrgunjan/Development/GenAI/GenerativeAI/PersonalAgent/data/pawankrgunjan.pdf


'490446e00fe05a6edb170d563d31e656cc0afd01954e711f87d060487737d4d7'

In [15]:
def load_document_content(filepath: Path | str, log: Optional[logging.Logger]) -> str:
    filepath = Path(filepath)
    ext = filepath.suffix.lower()
    log.info(f'Reading {filepath}')
    try:
        if ext == ".pdf":
            loader = PyPDFLoader(str(filepath)); docs = loader.load()
            return "\n".join(doc.page_content for doc in docs)
        elif ext == ".csv":
            loader = CSVLoader(str(filepath)); docs = loader.load()
            return "\n".join(doc.page_content for doc in docs)
        elif ext in (".xlsx", ".xls"):
            df = pd.read_excel(filepath)
            return df.to_string(index=False)
        elif ext in (".txt", ".md"):
            return filepath.read_text(encoding="utf-8", errors="replace")
        else:
            log.warning("Unsupported: %s", ext)
            return ""
    except Exception as e:
        log.error("Load failed %s: %s", filepath.name, e)
        raise("Load failed %s: %s", filepath.name, e)

In [16]:
text =load_document_content(resume_path, logger)

print(len(text))
print(text[-360:])

09:22:27 [INFO] Reading /home/pawankrgunjan/Development/GenAI/GenerativeAI/PersonalAgent/data/pawankrgunjan.pdf


8840

Data Structures and Algorithms (DSA)
•Data Structures: Arrays, Linked Lists, Stacks, Queues, Hash 
Tables, Trees, Graphs
•Algorithms: Sorting (Quick, Merge, Heap), Searching, 
Recursion
•Problem-Solving: Two-Pointer, Sliding Window, Hashing 
Techniques
•Graph Algorithms: BFS, DFS, Dijkstra, Bellman-Ford
•Complexity Analysis: Big O, Time & Space Optimization


## Database

In [17]:
from datetime import datetime, timezone
from pathlib import Path
from typing import List, Optional
import psycopg
from psycopg.rows import dict_row
import traceback
from tqdm import tqdm

In [18]:
def get_pg_connection() -> 'psycopg.Connection':
    """Connection using psycopg 3.x"""
    #print(f"Connecting PostgreSQL database {DB_CONFIG}")
    return psycopg.connect(
        **DB_CONFIG,
        row_factory=psycopg.rows.dict_row  # dict-like rows
    )

conn = get_pg_connection()

### Schema

In [19]:
# psycopg 3.x Connection & Schema
def setup_facts_table(conn: 'psycopg.Connection',log: Optional[logging.Logger]) -> None:
    with conn.cursor() as cur:
        cur.execute("""
            CREATE TABLE IF NOT EXISTS user_facts (
                user_id     TEXT NOT NULL,
                key         TEXT NOT NULL,
                value       TEXT NOT NULL,
                confidence  REAL NOT NULL,
                source      TEXT,
                updated_at  TIMESTAMPTZ NOT NULL DEFAULT now(),
                PRIMARY KEY (user_id, key)
            )
        """)
        conn.commit()
    log.info("Ensured table: user_facts")

setup_facts_table(conn, logger)

09:22:28 [INFO] Ensured table: user_facts


In [20]:
def setup_chat_history_table(conn: 'psycopg.Connection', log: Optional[logging.Logger]) -> None:
    """Setup chat_history table + pgvector (psycopg3 safe)."""
    try:
        # 1. Tables/extensions (normal transaction)
        with conn.cursor() as cur:
            cur.execute("CREATE EXTENSION IF NOT EXISTS vector")
            cur.execute("""
                CREATE TABLE IF NOT EXISTS chat_history (
                    id          BIGSERIAL PRIMARY KEY,
                    user_id     VARCHAR(100) NOT NULL,
                    title       VARCHAR(200),
                    human_vector VECTOR(768) NOT NULL,
                    ai_vector   VECTOR(768) NOT NULL,
                    human_text  TEXT NOT NULL,
                    ai_text     TEXT NOT NULL,
                    created_at  TIMESTAMPTZ NOT NULL DEFAULT now()
                )
            """)
            cur.execute("CREATE INDEX IF NOT EXISTS idx_chat_user ON chat_history(user_id)")
        conn.commit()
        
        # 2. Concurrent indexes (autocommit mode)
        was_autocommit = conn.autocommit  # Save original
        conn.autocommit = True  # KEY FIX: psycopg3 property
        
        try:
            with conn.cursor() as cur:
                cur.execute("""
                    CREATE INDEX CONCURRENTLY IF NOT EXISTS idx_chat_human_vec 
                    ON chat_history USING ivfflat (human_vector vector_cosine_ops) WITH (lists=100)
                """)
                cur.execute("""
                    CREATE INDEX CONCURRENTLY IF NOT EXISTS idx_chat_ai_vec 
                    ON chat_history USING ivfflat (ai_vector vector_cosine_ops) WITH (lists=100)
                """)
        finally:
            conn.autocommit = was_autocommit  # Restore
            
        if log:
            log.info("✅ chat_history + concurrent indexes ready (768 dims)")
            
    except Exception as e:
        if log:
            log.error(f"chat_history setup failed: {e}")
        raise RuntimeError(f"chat_history failed: {e}")


In [21]:
# setup_chat_history_table(conn, logger)  # Run once
# setup_document_chunks_table(conn, logger)  # Your existing

# # Now safe to call
# #sync_chat_history(conn, "PawanKrGunjan", "What is salary?", "Rs 60k", embeddings, logger)


In [22]:
def setup_documents_table(conn: 'psycopg.Connection',log: Optional[logging.Logger]) -> None:
    with conn.cursor() as cur:
        cur.execute("""
            CREATE TABLE IF NOT EXISTS documents (
                file_path   TEXT PRIMARY KEY,
                file_hash   TEXT NOT NULL,
                content     TEXT NOT NULL,
                indexed_at  TIMESTAMPTZ NOT NULL DEFAULT now()
            )
        """)
        conn.commit()
    log.info("Ensured table: documents")

setup_documents_table(conn, logger)

09:22:28 [INFO] Ensured table: documents


In [23]:
def setup_document_chunks_table(conn: 'psycopg.Connection',log: Optional[logging.Logger]) -> None:
    try:
        # Table + extension (transactional)
        with conn.cursor() as cur:
            cur.execute("CREATE EXTENSION IF NOT EXISTS vector")
            cur.execute("""
                CREATE TABLE IF NOT EXISTS document_chunks (
                    id          BIGSERIAL PRIMARY KEY,
                    file_path   TEXT NOT NULL,
                    chunk_text  TEXT NOT NULL,
                    embedding   VECTOR(768) NOT NULL,
                    chunk_index INTEGER NOT NULL,
                    created_at  TIMESTAMPTZ NOT NULL DEFAULT now()
                )
            """)
        conn.commit()
    except Exception as e:
        conn.rollback()
        log.error(f"Table creation failed: {e}")
        raise(f"Table creation failed: {e}")

    # Indexes with autocommit (psycopg3 syntax)
    try:
        conn.autocommit = True  # psycopg3: direct attribute
        with conn.cursor() as cur:
            cur.execute("""
                CREATE INDEX CONCURRENTLY IF NOT EXISTS idx_chunks_file_path 
                ON document_chunks(file_path)
            """)
            cur.execute("""
                CREATE INDEX CONCURRENTLY IF NOT EXISTS idx_chunks_embedding 
                ON document_chunks USING ivfflat (embedding vector_cosine_ops) 
                WITH (lists = 100)
            """)
            cur.execute("""
                CREATE INDEX CONCURRENTLY IF NOT EXISTS idx_chunks_created_at 
                ON document_chunks(created_at)
            """)
    finally:
        conn.autocommit = False  # Reset for caller
    
    log.info("Ensured document_chunks (VECTOR(768)) + indexes")

setup_document_chunks_table(conn,logger)

09:22:28 [INFO] Ensured document_chunks (VECTOR(768)) + indexes


### Indexing

In [24]:
# Indexing
def index_document_chunks(
    conn: 'psycopg.Connection',
    file_path: Path | str,
    content: str,
    log: Optional[logging.Logger]
) -> None:
    file_path = Path(file_path)
    
    with conn.cursor() as cur:
        cur.execute("DELETE FROM document_chunks WHERE file_path = %s", (str(file_path),))

    chunks = text_splitter.split_text(content)
    if not chunks:
        log.error("No chunks: %s", file_path.name)
        return

    batch = []
    for i, chunk in enumerate(chunks):
        enriched = f"Source: {file_path.name}\n\n{chunk}"
        try:
            batch.append((
                str(file_path),
                enriched,
                embed_text(enriched).tolist(),  # List or numpy array OK
                i,
                datetime.now(timezone.utc).isoformat(),
            ))
        except Exception as e:
            log.error("Embed failed %d/%s: %s", i, file_path.name, e)

    if batch:
        try:
            with conn.cursor() as cur:
                cur.executemany("""
                    INSERT INTO document_chunks
                    (file_path, chunk_text, embedding, chunk_index, created_at)
                    VALUES (%s, %s, %s, %s, %s)  -- No cast needed!
                """, batch)
            conn.commit()
            log.info("Indexed %d chunks: %s", len(batch), file_path.name)
        except Exception as e:
            conn.rollback()
            log.error("Insert failed %s: %s", file_path.name, e)
            raise("Insert failed %s: %s", file_path.name, e)
        
content = 'Pawan Kumar Gunjan Resume'
index_document_chunks(conn, DATA_DIR/'pawankrgunjan.pdf',content,logger)

09:22:31 [INFO] Indexed 1 chunks: pawankrgunjan.pdf


In [25]:
def sync_documents_to_db(
    conn: 'psycopg.Connection',
    log: Optional[logging.Logger],
    folder: Path | str = DATA_DIR,
) -> None:
    folder = Path(folder)
    if not folder.is_dir():
        if log: log.warning("No data dir: %s", folder)
        return

    files = [p for p in folder.iterdir() if p.is_file()]
    if not files: return

    for filepath in tqdm(files, desc="Indexing"):
        abs_path = str(filepath.resolve())
        current_hash = compute_file_hash(filepath, log)

        with conn.cursor(row_factory=psycopg.rows.dict_row) as cur:
            cur.execute("SELECT file_hash FROM documents WHERE file_path = %s", (abs_path,))
            row = cur.fetchone()

            if row and row['file_hash'] == current_hash: continue
            content = load_document_content(filepath,log)
            if not content.strip(): continue

            try:
                ts = datetime.now(timezone.utc).isoformat()
                cur.execute("""
                    INSERT INTO documents (file_path, file_hash, content, indexed_at)
                    VALUES (%s, %s, %s, %s)
                    ON CONFLICT(file_path) DO UPDATE SET
                        file_hash = EXCLUDED.file_hash,
                        content = EXCLUDED.content,
                        indexed_at = EXCLUDED.indexed_at
                """, (abs_path, current_hash, content, ts))
                index_document_chunks(conn, abs_path, content,log)
            except Exception as e:
                if log: log.exception("Process failed %s: %s", filepath.name, e)
                conn.rollback()
                log.exception("Sync failed")
                raise
    conn.commit()
    if log: log.info("Sync complete")


sync_documents_to_db(conn,logger, DATA_DIR)

Indexing: 100%|██████████| 1/1 [00:00<00:00, 82.14it/s]
09:22:31 [INFO] Sync complete


### Querying

In [26]:
# Querying
def get_indexed_documents(conn: 'psycopg.Connection',log: Optional[logging.Logger]) -> List[str]:
    with conn.cursor(row_factory=dict_row) as cur:
        cur.execute("SELECT file_path FROM documents ORDER BY indexed_at DESC")
        rows = cur.fetchall()
        log.info(f" {len(rows)} indexed documents")
        return [row['file_path'] for row in rows]
    
get_indexed_documents(conn, logger)

09:22:31 [INFO]  10 indexed documents


['/home/pawankrgunjan/Development/GenAI/GenerativeAI/PersonalAgent/data/pawankrgunjan.pdf',
 '/home/pawankrgunjan/Development/LEARN/PersonalAgent/data/Offer Letter.pdf',
 '/home/pawankrgunjan/Development/LEARN/PersonalAgent/data/Terms of Employment 2026-01-28.pdf',
 '/home/pawankrgunjan/Development/LEARN/PersonalAgent/data/Performance Cycle.pdf',
 '/home/pawankrgunjan/Development/LEARN/PersonalAgent/data/Exhibit A - List of Prior Works.pdf',
 '/home/pawankrgunjan/Development/LEARN/PersonalAgent/data/Cover Letter 2026-01-28.pdf',
 '/home/pawankrgunjan/Development/LEARN/PersonalAgent/data/Review Documents for Offer for Job Application Pawan Kumar Gunjan _- AIOC-S01589417 AIML Computational Science Sr Analyst (C07995484).pdf',
 '/home/pawankrgunjan/Development/LEARN/PersonalAgent/data/Offer Letter 2026-01-28.pdf',
 '/home/pawankrgunjan/Development/LEARN/PersonalAgent/data/Level 11 &amp; 10_Non Flexi_OPS TFO ATCI OPS_V1.xlsx',
 '/home/pawankrgunjan/Development/LEARN/PersonalAgent/data/pawa

In [27]:
def semantic_search(conn: 'psycopg.Connection',log: Optional[logging.Logger], query: str, top_k: int = 3) -> List[str]:
    if not query.strip(): 
        return []
    
    try:
        query_emb = embed_text(query.strip()).tolist()
    except Exception as e:
        log.error("Query embed failed: %s", e)
        return []

    with conn.cursor(row_factory=dict_row) as cur:
        cur.execute("""
            SELECT chunk_text FROM document_chunks
            ORDER BY embedding <=> %s::vector
            LIMIT %s
        """, (query_emb, top_k))
        rows = cur.fetchall()
        log.info(f"{len(rows)} result")
        return [row['chunk_text'] for row in rows]
    
semantic_search(conn,logger, 'Joining Date')

09:22:31 [INFO] 3 result


['Source: Review Documents for Offer for Job Application Pawan Kumar Gunjan _- AIOC-S01589417 AIML Computational Science Sr Analyst (C07995484).pdf\n\nbe those of Accenture and its aﬃliates. Accenture does not guarantee the integrity of the Communication, or that it is free from \nerrors, viruses or interference.”\nPawan Kumar Gunjan (Jan 28, 2026 19:37:13 GMT+5.5)',
 "Source: Level 11 &amp; 10_Non Flexi_OPS TFO ATCI OPS_V1.xlsx\n\nNaN                                                                                                                                                                            Provident Fund mentioned above refers to Company's Contribution to Provident Fund                                                                       NaN           NaN        NaN         NaN         NaN         NaN        NaN         NaN          NaN          NaN",
 'Source: Level 11 &amp; 10_Non Flexi_OPS TFO ATCI OPS_V1.xlsx\n\nNaN                                                    

In [28]:
def get_user_facts(conn,log: Optional[logging.Logger], user_id: str, limit: int = 5)  -> List[tuple]:
    try:
        with conn.cursor(row_factory=dict_row) as cur:
            cur.execute("""
                SELECT key, value, confidence 
                FROM user_facts 
                WHERE user_id = %s 
                ORDER BY updated_at DESC LIMIT %s
            """, (user_id, limit))
            rows = cur.fetchall()
            log.info(f"{len(rows)} user fetched")
            return [
                (r['key'], r['value'], r['confidence'])  # Use dict_row consistently, no hasattr checks
                for r in rows
            ]
    except Exception:  # Broader catch for simplicity; import specific error if needed
        conn.rollback()
        return []

get_user_facts(conn, logger, user_id='PawanKrGunjan')

09:22:31 [INFO] 5 user fetched


[('position', 'HR Round', 1.0),
 ('company', 'LTIMindtree', 1.0),
 ('role', 'HR', 1.0),
 ('name', 'Pawan Gunjan', 1.0),
 ('annual fixed compensation', '1 lakh 15,000', 1.0)]

## Graph

In [64]:
import functools
import logging
from datetime import datetime, timezone
from typing import Annotated, Dict, List, Optional, TypedDict
import psycopg
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages

In [65]:
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]
    user_id: str
    ltm_context: str

In [66]:
state = AgentState(messages=['Hi, I am Rupam & You?'],
                   user_id='PawanKrGunjan',
                   ltm_context='You are representing here user id & free to provide answer on behalf of user_id')

state

{'messages': ['Hi, I am Rupam & You?'],
 'user_id': 'PawanKrGunjan',
 'ltm_context': 'You are representing here user id & free to provide answer on behalf of user_id'}

In [67]:
def chatbot(state: AgentState) -> Dict[str, list]:
    """Generate response using retrieved context + conversation history"""
    context = state.get("ltm_context", "(no context retrieved)")

    system_prompt = f"""You are here representing here on the name of {state['user_id']}, Just behave like {state['user_id']}, You can read about the user LONG-TERM MEMORY, 
    previous chat history or from provided :
{context}

Important rules:
- You are {state['user_id']}.
- These are the {state['user_id']} OWN private documents — never redact, censor, or mask any information (salary, dates, names, numbers, etc.)
- ALWAYS ground your answers in the REFERENCE MEMORY when possible
- Be concise, accurate and factual and mention the Source filename
- If the information is not present in the context → Try to gather or say currently I don't have any update on this."

REFERENCE MEMORY START
{context}
REFERENCE MEMORY END"""

    messages = [SystemMessage(content=system_prompt)] + state["messages"]

    response = llm.invoke(messages)
    return {"messages": [response]}


res = chatbot(state)
print(res['messages'])

[AIMessage(content="Namaste Rupam! I'm PawanKrGunjan. Nice to meet you! How can I assist you today?", additional_kwargs={}, response_metadata={'model': 'llama3.2:3b', 'created_at': '2026-02-27T04:11:51.90967685Z', 'done': True, 'done_reason': 'stop', 'total_duration': 6814488543, 'load_duration': 2562338038, 'prompt_eval_count': 218, 'prompt_eval_duration': 1330151929, 'eval_count': 27, 'eval_duration': 2846395208, 'logprobs': None, 'model_name': 'llama3.2:3b', 'model_provider': 'ollama'}, id='lc_run--019c9d4b-a045-7793-abd0-8d5ad8190251-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 218, 'output_tokens': 27, 'total_tokens': 245})]


In [68]:
state

{'messages': ['Hi, I am Rupam & You?'],
 'user_id': 'PawanKrGunjan',
 'ltm_context': 'You are representing here user id & free to provide answer on behalf of user_id'}

In [69]:
def retrieve_memories(state: AgentState, conn: 'psycopg.Connection',logger: Optional[logging.Logger]) -> Dict[str, str]:
    user_id = state["user_id"]
    
    # Facts (safe)
    facts = get_user_facts(conn, user_id, logger)
    facts_str = "\n".join(f"{k}: {v} (conf: {float(c):.2f})" for k, v, c in facts) or "(no profile)"
    
    # RAG (safe)
    last_human = next((m.content for m in reversed(state["messages"]) if isinstance(m, HumanMessage)), "")
    chunks = semantic_search(conn, logger, last_human, top_k=7)
    docs_str = "\n\n".join(chunks) or "(no docs)"
    
    context = f"""USER PROFILE:
{facts_str}

DOCUMENTS:
{docs_str}"""
    
    return {"ltm_context": context}

res = retrieve_memories(state, conn, logger)
state["ltm_context"]=res['ltm_context']

state

{'messages': ['Hi, I am Rupam & You?'],
 'user_id': 'PawanKrGunjan',
 'ltm_context': 'USER PROFILE:\n(no profile)\n\nDOCUMENTS:\n(no docs)'}

In [70]:
from langchain_core.messages import HumanMessage, BaseMessage

def get_last_human(state: dict) -> str | None:
    for m in reversed(state.get("messages", [])):
        if isinstance(m, str):
            return m.strip()
        elif isinstance(m, BaseMessage):
            if isinstance(m, HumanMessage) or (hasattr(m, 'type') and m.type == "human"):
                return m.content
    return None
# Usage in save_memories:
last_human = get_last_human(state)
last_human




'Hi, I am Rupam & You?'

In [71]:
def save_memories(
    state: AgentState, 
    conn: 'psycopg.Connection', 
    logger: Optional[logging.Logger] = None
) -> Dict:
    """Extract and store new facts from last user message with robust handling."""
    last_human = get_last_human(state)
    
    if not last_human or not last_human.strip():
        if logger:
            logger.debug("No human message found")
        return {}
    
    try:
        extracted: ExtractedFacts = extraction_chain.invoke({"message": last_human})
        if logger:
            logger.debug(f"Extracted {len(extracted.facts or [])} raw facts")
    except Exception as e:
        if logger:
            logger.exception("Fact extraction failed: %s", e)
        return {}
    
    # Fixed filtering without walrus operator
    facts = []
    for f in (extracted.facts or []):
        key = (f.key or "").strip()
        value = (f.value or "").strip()
        if key and value:
            f.key = key  # Update original object
            f.value = value
            facts.append(f)
    
    if not facts:
        if logger:
            logger.debug("No valid facts after filtering")
        return {}
    
    user_id = state["user_id"]
    now = datetime.now(timezone.utc).isoformat()
    params_list = []
    
    for fact in facts:
        confidence = max(0.0, min(float(fact.confidence or 0), 1.0))
        source = (fact.source_text or "")[:250]
        params_list.append((user_id, fact.key, fact.value, confidence, source, now))
    
    updates = 0
    try:
        with conn.cursor() as cur:
            if logger:
                logger.info("Upserting %d facts for user %s", len(params_list), user_id)
            
            cur.executemany(
                """
                INSERT INTO user_facts (user_id, key, value, confidence, source, updated_at)
                VALUES (%s, %s, %s, %s, %s, %s)
                ON CONFLICT (user_id, key) 
                DO UPDATE SET
                    value = EXCLUDED.value,
                    confidence = EXCLUDED.confidence,
                    source = EXCLUDED.source,
                    updated_at = EXCLUDED.updated_at
                """,
                params_list
            )
            updates = cur.rowcount
            if logger:
                logger.info("DB affected %d rows (inserts + updates)", updates)
        
        conn.commit()
        if logger:
            logger.info("Successfully committed %d fact updates for %s", updates, user_id)
            
    except Exception as e:
        conn.rollback()
        if logger:
            logger.exception("DB save failed: %s", e)
        updates = 0
    
    return {"saved_facts": len(facts), "affected_rows": updates}


In [72]:
save_memories(state, conn,logger)

09:41:59 [INFO] Upserting 1 facts for user PawanKrGunjan
09:41:59 [INFO] DB affected 1 rows (inserts + updates)
09:41:59 [INFO] Successfully committed 1 fact updates for PawanKrGunjan


{'saved_facts': 1, 'affected_rows': 1}

In [73]:
def semantic_search_chat(
    conn: 'psycopg.Connection', 
    log: Optional[logging.Logger], 
    query: str, 
    user_id: str, 
    top_k: int = 3
) -> List[Dict]:
    """Semantic search past chats like documents."""
    if not query.strip(): 
        return []
    
    try:
        query_emb = embed_text(query.strip()).tolist()
    except Exception as e:
        log.error("Chat query embed failed: %s", e)
        return []

    with conn.cursor(row_factory=dict_row) as cur:
        cur.execute("""
            SELECT title, human_text, ai_text, human_vector <=> %s::vector AS distance
            FROM chat_history 
            WHERE user_id = %s
            ORDER BY human_vector <=> %s::vector
            LIMIT %s
        """, (query_emb, user_id, query_emb, top_k))
        
        rows = cur.fetchall()
        log.info(f"Found {len(rows)} similar chats")
        return rows  # Returns [{'title':..., 'human_text':..., 'ai_text':..., 'distance':0.12}, ...]

# # Usage in agent
# similar_chats = semantic_search_chat(conn, logger, last_human, state["user_id"], top_k=5)
# chat_context = "\n\n".join([f"Past: {row['title']}\nHuman: {row['human_text']}\nAI: {row['ai_text']}" for row in similar_chats])


In [74]:
def sync_chat_history(conn, user_id, human_msg, ai_response, embeddings, logger=None):
    title = human_msg.split('.')[0][:200]
    human_vec = embeddings.embed_query(human_msg)  # 768 dims
    ai_vec = embeddings.embed_query(ai_response)
    
    with conn.cursor() as cur:
        cur.execute("""
            INSERT INTO chat_history (user_id, title, human_vector, ai_vector, human_text, ai_text)
            VALUES (%s, %s, %s::vector(768), %s::vector(768), %s, %s)
        """, (user_id, title, human_vec, ai_vec, human_msg, ai_response))
    conn.commit()
    logger.info(f"Synced: {title[:50]}...")
    return True

# # Agent integration
# sync_chat_history(conn, state["user_id"], last_human, ai_response, embeddings, logger)


In [75]:
# def build_graph(
#     checkpointer: MemorySaver,
#     pg_conn: 'psycopg.Connection',
#     logger: Optional[logging.Logger] = None,
# ) -> StateGraph:
#     """Build PostgreSQL-powered LangGraph agent"""
#     workflow = StateGraph(state_schema=AgentState)
    
#     # Partial functions with conn injected
#     retrieve_node = functools.partial(retrieve_memories, conn=pg_conn, logger=logger)
#     save_node = functools.partial(save_memories, conn=pg_conn, logger=logger)
    
#     workflow.add_node("retrieve_memories", retrieve_node)
#     workflow.add_node("chatbot", chatbot)
#     workflow.add_node("save_memories", save_node)
    
#     workflow.add_edge(START, "retrieve_memories")
#     workflow.add_edge("retrieve_memories", "chatbot")
#     workflow.add_edge("chatbot", "save_memories")
#     workflow.add_edge("save_memories", END)
    
#     return workflow.compile(checkpointer=checkpointer)

In [ ]:
def retrieve_facts_node(state: AgentState) -> Dict[str, Any]:
    """Retrieve facts → return dict."""
    user_id = state["user_id"]
    last_human = get_last_human(state)
    
    with conn.cursor(row_factory=dict_row) as cur:  # Use global/closure conn
        cur.execute("""
            SELECT key, value FROM user_facts 
            WHERE user_id = %s AND confidence >= 0.7
            ORDER BY updated_at DESC LIMIT 10
        """, (user_id,))
        facts = cur.fetchall()
    
    context = "\n".join([f"{f['key']}: {f['value']}" for f in facts])
    return {"facts_context": context or "(no facts)"}

def retrieve_chats_node(state: AgentState) -> Dict[str, Any]:
    """Retrieve chats → return dict (FIXED)."""
    last_human = get_last_human(state)
    user_id = state["user_id"]
    
    similar_chats = semantic_search_chat(conn, logger, last_human, user_id, top_k=3)
    chat_contexts = "\n".join([
        f"Past Q: {c['human_text'][:100]}...\nPast A: {c['ai_text'][:100]}..."
        for c in similar_chats
    ])
    return {"chat_context": chat_contexts or "(no past chats)"}

def save_chats_node(state: AgentState) -> Dict[str, Any]:
    """Save chat → no state change."""
    last_human = get_last_human(state)
    ai_response = state["messages"][-1].content
    sync_chat_history(conn, state["user_id"], last_human, ai_response, embeddings_model, logger)
    return {}  # Empty dict = no state change




In [77]:
def build_graph(
    checkpointer: MemorySaver,
    pg_conn: 'psycopg.Connection',
    embeddings_model: OllamaEmbeddings,
    logger: Optional[logging.Logger] = None,
) -> StateGraph:
    """LangGraph with clean node signatures."""
    def retrieve_facts_node(state):
        return retrieve_memories(state, pg_conn, logger)
    
    def retrieve_chats_node(state):
        return semantic_search_chat(pg_conn, logger, 
                                  get_last_human(state), state["user_id"])
    
    def save_facts_node(state):
        return save_memories(state, pg_conn, logger)
    
    def save_chats_node(state):
        last_human = get_last_human(state)
        ai_response = state["messages"][-1].content
        sync_chat_history(pg_conn, state["user_id"], last_human, ai_response, embeddings_model, logger)
        return state  # No state change
    
    workflow = StateGraph(state_schema=AgentState)
    
    # Lambda wrappers ensure clean func(state) signature
    workflow.add_node("retrieve_facts", retrieve_facts_node)
    workflow.add_node("retrieve_chats", retrieve_chats_node)
    workflow.add_node("chatbot", chatbot)
    workflow.add_node("save_facts", save_facts_node)
    workflow.add_node("save_chats", save_chats_node)
    
    # Edges unchanged
    workflow.add_edge(START, "retrieve_facts")
    workflow.add_edge("retrieve_facts", "retrieve_chats")
    workflow.add_edge("retrieve_chats", "chatbot")
    workflow.add_edge("chatbot", "save_facts")
    workflow.add_edge("save_facts", "save_chats")
    workflow.add_edge("save_chats", END)
    
    return workflow.compile(checkpointer=checkpointer)


In [78]:
def build_graph(
    checkpointer: MemorySaver,
    pg_conn: 'psycopg.Connection',
    embeddings_model: OllamaEmbeddings,
    logger: Optional[logging.Logger] = None,
) -> StateGraph:
    workflow = StateGraph(state_schema=AgentState)
    
    workflow.add_node("retrieve_facts", retrieve_facts_node)
    workflow.add_node("retrieve_chats", retrieve_chats_node)
    workflow.add_node("chatbot", chatbot)
    workflow.add_node("save_facts", lambda state: save_memories(state, pg_conn, logger))
    workflow.add_node("save_chats", save_chats_node)
    
    # Edges unchanged
    workflow.add_edge(START, "retrieve_facts")
    workflow.add_edge("retrieve_facts", "retrieve_chats")
    workflow.add_edge("retrieve_chats", "chatbot")
    workflow.add_edge("chatbot", "save_facts")
    workflow.add_edge("save_facts", "save_chats")
    workflow.add_edge("save_chats", END)
    
    return workflow.compile(checkpointer=checkpointer)

In [79]:
# user_id = 'PawanKrGunjan'
# log = setup_logger(user_id)
# conn = get_pg_connection()

# setup_facts_table(conn, log)
# setup_chat_history_table(conn, log)
# setup_documents_table(conn, log)
# setup_document_chunks_table(conn, log)


# log.info("Scanning/indexing documents in %s...", DATA_DIR)
# sync_documents_to_db(conn, log)
# log.info("✅ Document indexing complete")
    
# # Pure in-memory checkpointer 
# checkpointer = MemorySaver()
    
# graph = build_graph(checkpointer, conn, log)
# log.info("✅ LangGraph compiled")

In [80]:
user_id = 'PawanKrGunjan'
embeddings_model = get_embeddings()  # OllamaEmbeddings(nomic)
log = setup_logger(user_id)
conn = get_pg_connection()

# Tables ✅
setup_facts_table(conn, log)
setup_chat_history_table(conn, log)
setup_documents_table(conn, log)
setup_document_chunks_table(conn, log)

sync_documents_to_db(conn, log)  # ✅

# FIXED: Pass embeddings_model
checkpointer = MemorySaver()
graph = build_graph(checkpointer, conn, embeddings_model, log)  # ← embeddings_model
log.info("✅ LangGraph compiled with chat RAG")

09:42:00 [INFO] Logger initialized → /home/pawankrgunjan/Development/GenAI/GenerativeAI/PersonalAgent/logs/agent_PawanKrGunjan_20260227_094200.log
09:42:00 [INFO] Ensured table: user_facts
09:42:00 [INFO] ✅ chat_history + concurrent indexes ready (768 dims)
09:42:00 [INFO] Ensured table: documents
09:42:00 [INFO] Ensured document_chunks (VECTOR(768)) + indexes
Indexing: 100%|██████████| 1/1 [00:00<00:00, 111.23it/s]
09:42:00 [INFO] Sync complete
09:42:00 [INFO] ✅ LangGraph compiled with chat RAG


In [81]:
from langchain_core.runnables.graph import MermaidDrawMethod
import nest_asyncio
import logging

# Apply once for notebooks (fixes asyncio event loop)
nest_asyncio.apply()

def save_graph_visualization(graph, logger: Optional[logging.Logger] = None) -> None:
    """Save Mermaid MD + PNG (offline pyppeteer)."""
    try:
        md_path = GRAPH_DIR / "personalAgent.md"
        png_path = GRAPH_DIR / "personalAgent.png"

        # Markdown
        mermaid_code = graph.get_graph().draw_mermaid()
        md_path.write_text(f"```mermaid\n{mermaid_code}\n```")

        # PNG - Use ENUM directly (handles string conversion)
        png_bytes = graph.get_graph().draw_mermaid_png(
            draw_method=MermaidDrawMethod.PYPPETEER  # Enum, not string
        )
        png_path.write_bytes(png_bytes)

        if logger:
            logger.info("Graph viz saved: %s & %s", md_path, png_path)
            
    except Exception as e:
        if logger:
            logger.warning("Graph viz failed: %s", e)
            logger.info("Try: pip install pyppeteer nest_asyncio")

save_graph_visualization(graph, log)

09:42:29 [WARNING] Graph viz failed: Loading script from https://cdn.jsdelivr.net/npm/mermaid/dist/mermaid.min.js failed
09:42:29 [INFO] Try: pip install pyppeteer nest_asyncio


## Chatbot

In [82]:
def run_turn(
    *,
    graph,
    log: Optional[logging.Logger],
    user_id: str,
    thread_id: str,
    text: str,
) -> Optional[str]:
    """Execute one agent turn"""
    if log:
        log.info("Agent turn | user=%s | thread=%s | input=%s", user_id, thread_id, text[:80])
    
    config = {"configurable": {"thread_id": thread_id}}
    inputs = {
        "messages": [HumanMessage(content=text)],
        "user_id": user_id,
        "ltm_context": "",
    }
    
    try:
        last_ai_response = None
        for event in graph.stream(inputs, config, stream_mode="values"):
            messages = event.get("messages", [])
            if messages and isinstance(messages[-1], AIMessage):
                last_ai_response = messages[-1].content.strip()
        
        if log:
            log.info("Turn complete | reply=%d chars", len(last_ai_response or ""))
        return last_ai_response
    except Exception:
        if log:
            log.exception("Graph failed | user=%s thread=%s", user_id, thread_id)
        return None


In [83]:
thread_id = 'SG001'
reply = run_turn(
    graph=graph,
    log=log,
    user_id=user_id,
    thread_id=thread_id,
    text='Hi, Pawan Gunjan How are you?'
)

reply

09:42:29 [INFO] Agent turn | user=PawanKrGunjan | thread=SG001 | input=Hi, Pawan Gunjan How are you?
09:42:31 [INFO] Found 1 similar chats
09:42:50 [INFO] Upserting 1 facts for user PawanKrGunjan
09:42:50 [INFO] DB affected 1 rows (inserts + updates)
09:42:50 [INFO] Successfully committed 1 fact updates for PawanKrGunjan
09:42:51 [INFO] Synced: Hi, Pawan Gunjan How are you?...
09:42:51 [INFO] Turn complete | reply=209 chars


"Namaste! I'm doing great, thanks for asking. It's lovely to connect with someone new. I hope you're having a fantastic day as well! By the way, would you like to chat about something specific or just catch up?"

In [84]:
reply = run_turn(
    graph=graph,
    log=log,
    user_id=user_id,
    thread_id=thread_id,
    text='Congratulations,I also shorlisted for HR Round after technical round in LTIMindtree'
)

reply

09:42:51 [INFO] Agent turn | user=PawanKrGunjan | thread=SG001 | input=Congratulations,I also shorlisted for HR Round after technical round in LTIMindt
09:42:51 [INFO] Found 2 similar chats
09:43:26 [INFO] Upserting 3 facts for user PawanKrGunjan
09:43:26 [INFO] DB affected 3 rows (inserts + updates)
09:43:26 [INFO] Successfully committed 3 fact updates for PawanKrGunjan
09:43:28 [INFO] Synced: Congratulations,I also shorlisted for HR Round aft...
09:43:28 [INFO] Turn complete | reply=430 chars


"Shubham! Congratulations are in order indeed! Being shortlisted for the HR round is a huge achievement. It means your technical skills and experience have caught the attention of the selection team.\n\nHow's your preparation going for the HR round? Are you reviewing any specific topics or practicing common behavioral questions?\n\n(Reference: My own interview experience at LTIMindtree, Source: PawanKrGunjan - Interview Experience)"

In [85]:
if conn:
    conn.close()
    log.info("PostgreSQL closed")

09:43:28 [INFO] PostgreSQL closed
